# Demo 01 — Single-Table CTGAN

This notebook trains a **Conditional Tabular GAN (CTGAN)** on a single mixed-type table and samples synthetic rows. It demonstrates the core SynthoHive workflow:

1. Build or load real data
2. Define metadata (primary keys, PII columns, high-cardinality columns)
3. Fit a CTGAN model
4. Sample synthetic rows

No external data files or Spark are required — the demo generates its own training data.

In [ ]:
import numpy as np
import pandas as pd

from syntho_hive.interface.config import Metadata
from syntho_hive.core.models.ctgan import CTGAN

## 1. Build Training Data

We create a 600-row table with mixed types: integer IDs, ages, continuous spend, categorical city and channel, and a float loyalty score.

In [ ]:
def build_training_data(num_rows: int = 600) -> pd.DataFrame:
    """Create a small, mixed-type table to keep the demo self-contained."""
    rng = np.random.default_rng(42)
    cities = ["NYC", "SF", "SEA", "CHI", "DAL", "MIA"]
    channels = ["web", "retail", "partner"]

    df = pd.DataFrame(
        {
            "customer_id": np.arange(1, num_rows + 1),
            "age": rng.integers(18, 75, size=num_rows),
            "annual_spend": rng.normal(55000, 15000, size=num_rows).clip(5000, 150000).round(2),
            "city": rng.choice(cities, p=[0.28, 0.22, 0.18, 0.14, 0.1, 0.08], size=num_rows),
            "signup_channel": rng.choice(channels, p=[0.6, 0.3, 0.1], size=num_rows),
            "loyalty_score": rng.uniform(0, 1, size=num_rows).round(4),
        }
    )
    return df

train_df = build_training_data()
print(f"Training data shape: {train_df.shape}")
train_df.head()

## 2. Configure Metadata

We tell SynthoHive which column is the primary key and which columns contain PII or have high cardinality. The PK column (`customer_id`) is excluded from modeling — it will be re-assigned after generation.

In [ ]:
meta = Metadata()
meta.add_table(
    name="customers",
    pk="customer_id",
    pii_cols=["customer_id"],
    high_cardinality_cols=["city"],
)

## 3. Train the CTGAN

We use a small architecture (128-unit layers, 64-dim embeddings) and only 5 epochs to keep the demo fast. In production you would increase `epochs`, `batch_size`, and layer sizes.

In [ ]:
model = CTGAN(
    meta,
    batch_size=128,
    epochs=5,
    embedding_dim=64,
    generator_dim=(128, 128),
    discriminator_dim=(128, 128),
    device="cpu",
)
model.fit(train_df, table_name="customers")
print("Training complete.")

## 4. Sample Synthetic Rows

We generate 200 synthetic rows and re-attach sequential `customer_id` values (since the PK was excluded from modeling).

In [ ]:
synthetic_df = model.sample(200)
synthetic_df.insert(0, "customer_id", range(1, len(synthetic_df) + 1))

print(f"Synthetic data shape: {synthetic_df.shape}")
synthetic_df.head(10)

## 5. Quick Comparison

A side-by-side look at summary statistics for the real and synthetic data.

In [ ]:
print("=== Real Data ===")
print(train_df.describe())
print("\n=== Synthetic Data ===")
print(synthetic_df.describe())